# CELL 1 :Imports for NumPy and Matplotlib for all plots 

In [2]:
import csv
import time
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(suppress=True, precision=4)
rng = np.random.default_rng(42)

CSV_PATH = "Education in General.csv"
print("NumPy", np.__version__)

NumPy 2.3.3


# CELL 2 : Load the CSV and drop the two empty trailing columns. utf-8-sig removes the BOM.

In [3]:
MISSING_TOKENS = {"#N/B", "", "NA", "N/A", "nan", "NaN", "..", "-"}

with open(CSV_PATH, newline="", encoding="utf-8-sig") as f:
    rows = list(csv.reader(f))

header, data_rows = rows[0], rows[1:]
print("Raw shape: %d rows x %d columns" % (len(data_rows), len(header)))

keep = [j for j, name in enumerate(header) if name.strip() and not name.startswith("Unnamed")]
header    = [header[j] for j in keep]
data_rows = [[r[j] if j < len(r) else "" for j in keep] for r in data_rows]
n_rows = len(data_rows)
print("After dropping empty columns: %d columns" % len(header))
for h in header: print("  -", h)

Raw shape: 756 rows x 13 columns
After dropping empty columns: 11 columns
  - ISO_Code
  - Country
  - Year
  - School life expectancy, primary to tertiary, male (years)
  - School life expectancy, primary to tertiary, female (years)
  - Government expenditure on primary education, US$ (millions)
  - Government expenditure on secondary education, US$ (millions)
  - Government expenditure on tertiary education, US$ (millions)
  - Government expenditure on primary education as a percentage of GDP (%)
  - Government expenditure on secondary education as a percentage of GDP (%)
  - Government expenditure on tertiary education as a percentage of GDP (%)


# CELL 3 : Detect non-numeric columns automatically (ISO_Code, Country come out as categorical).

In [4]:
def is_number(s):
    s = s.strip()
    if s in MISSING_TOKENS:
        return True
    try:
        float(s); return True
    except ValueError:
        return False

col_numeric = []
for j in range(len(header)):
    col_numeric.append(all(is_number(data_rows[i][j]) for i in range(n_rows)))

numeric_cols     = [header[j] for j in range(len(header)) if col_numeric[j]]
categorical_cols = [header[j] for j in range(len(header)) if not col_numeric[j]]
print("NON-NUMERIC (categorical) columns :", categorical_cols)
print("NUMERIC columns                   :", len(numeric_cols), "found")

NON-NUMERIC (categorical) columns : ['ISO_Code', 'Country']
NUMERIC columns                   : 9 found


# CELL 4 : Encode the categorical columns with label encoding. Country is an identifier, so we encode it but keep it only as a label, not as a PCA feature.

In [5]:
col = {name: j for j, name in enumerate(header)}

def label_encode(values):
    cats = sorted(set(values))
    mapping = {c: i for i, c in enumerate(cats)}
    return np.array([mapping[v] for v in values]), mapping

countries = [data_rows[i][col["Country"]] for i in range(n_rows)]
country_codes, country_map = label_encode(countries)
years = np.array([int(float(data_rows[i][col["Year"]])) for i in range(n_rows)])

period = np.select([years <= 2014, years <= 2019], [0, 1], default=2)
period_names = ["2010–2014", "2015–2019", "2020–2023"]

print("Label-encoded %d countries (codes 0..%d). Sample:" % (len(country_map), len(country_map)-1))
for c in list(country_map)[:5]:
    print("   %-15s -> %d" % (c, country_map[c]))

Label-encoded 54 countries (codes 0..53). Sample:
   Algeria         -> 0
   Angola          -> 1
   Benin           -> 2
   Botswana        -> 3
   Burkina Faso    -> 4


# CELL 5 : Build the 8-feature numeric matrix X and turn '#N/B' markers into NaN. Year is excluded.

In [6]:
feature_cols = [c for c in numeric_cols if c != "Year"]
feat_idx = [col[c] for c in feature_cols]

def to_float(s):
    s = s.strip()
    return np.nan if s in MISSING_TOKENS else float(s)

X_raw = np.array([[to_float(data_rows[i][j]) for j in feat_idx] for i in range(n_rows)], float)

print("Feature matrix X: %d rows x %d features\n" % X_raw.shape)
print("Missing values per feature (was '#N/B'):")
for name, m in zip(feature_cols, np.isnan(X_raw).sum(0)):
    print("   %3d  (%.0f%%)  %s" % (m, 100*m/n_rows, name))

Feature matrix X: 756 rows x 8 features

Missing values per feature (was '#N/B'):
   496  (66%)  School life expectancy, primary to tertiary, male (years)
   496  (66%)  School life expectancy, primary to tertiary, female (years)
   504  (67%)  Government expenditure on primary education, US$ (millions)
   500  (66%)  Government expenditure on secondary education, US$ (millions)
   509  (67%)  Government expenditure on tertiary education, US$ (millions)
   450  (60%)  Government expenditure on primary education as a percentage of GDP (%)
   443  (59%)  Government expenditure on secondary education as a percentage of GDP (%)
   448  (59%)  Government expenditure on tertiary education as a percentage of GDP (%)


# CELL 6 : Impute missing values with the column MEDIAN (robust to the outlier % -of-GDP errors).

In [7]:
def median_impute(X):
    X = X.copy()
    med = np.nanmedian(X, axis=0)
    nan_r, nan_c = np.where(np.isnan(X))
    X[nan_r, nan_c] = med[nan_c]
    return X, med

X_imp, medians = median_impute(X_raw)
print("NaNs before:", int(np.isnan(X_raw).sum()), "| after imputation:", int(np.isnan(X_imp).sum()))

NaNs before: 3846 | after imputation: 0


# CELL 7 : Standardize to mean 0, std 1. Needed because features have different units  (years vs US$ millions vs %); otherwise big-number columns would dominate PCA.

In [8]:
def standardize(X):
    mu = X.mean(0)
    sd = X.std(0, ddof=0); sd[sd == 0] = 1.0
    return (X - mu) / sd, mu, sd

X_std, mu, sd = standardize(X_imp)
print("Per-feature mean after scaling (≈0):", X_std.mean(0))
print("Per-feature std  after scaling (≈1):", X_std.std(0))

Per-feature mean after scaling (≈0): [ 0. -0.  0. -0. -0. -0. -0. -0.]
Per-feature std  after scaling (≈1): [1. 1. 1. 1. 1. 1. 1. 1.]


# CELL 8 : TASK 1: PCA from scratch. Center -> covariance matrix -> eigen-decomposition (eigh) -> sort eigenvalues descending -> project. Eigenvectors = principal components, eigenvalues = variance along each.

In [9]:
def pca_fit(X, n_components=None):
    '''PCA from scratch. Returns (scores, eigenvalues, eigenvectors, covariance).'''
    n = X.shape[0]
    Xc  = X - X.mean(0)                       # 1. center
    cov = (Xc.T @ Xc) / (n - 1)               # 2. covariance matrix
    eigvals, eigvecs = np.linalg.eigh(cov)    # 3. eigh for symmetric matrices
    order   = np.argsort(eigvals)[::-1]       # 4. sort DESCENDING
    eigvals, eigvecs = eigvals[order], eigvecs[:, order]
    if n_components is not None:
        eigvals, eigvecs = eigvals[:n_components], eigvecs[:, :n_components]
    scores = Xc @ eigvecs                     # 5. project
    return scores, eigvals, eigvecs, cov

scores, eigvals, eigvecs, cov = pca_fit(X_std)

print("Covariance matrix shape:", cov.shape)
print("\nEigenvalues (variance per component, descending):")
print(eigvals)
print("\nSorted descending? ->", bool(np.all(np.diff(eigvals) <= 1e-9)))

Covariance matrix shape: (8, 8)

Eigenvalues (variance per component, descending):
[3.0634 3.0008 1.7339 0.1371 0.0653 0.0101 0.0001 0.    ]

Sorted descending? -> True
